# 06 · Regresión y clasificación

En este notebook vamos a trabajar con **dos tipos de problemas supervisados**:

1. **Regresión** — predecir un valor continuo (precio de una vivienda).
2. **Clasificación** — predecir una categoría (especie de flor, tumor benigno/maligno).

Cada sección sigue el mismo flujo didáctico:

```
datos → exploración → split → entrenar → predecir → métricas → gráfico
```

### Preparación
Si trabajas en local, instala dependencias:

In [ ]:
# !pip install numpy pandas matplotlib scikit-learn

---

## Sección 1 · Regresión lineal

**Objetivo:** predecir el `precio` de una vivienda a partir de su `superficie` y `nº de habitaciones`.

Usamos un dataset sintético con poco ruido para que el modelo lineal funcione muy bien y el gráfico se vea claro.

### 1.1 · Imports y datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_array, y_array = make_regression(
    n_samples=200,
    n_features=2,
    n_informative=2,
    noise=15,
    random_state=42,
)

df = pd.DataFrame(X_array, columns=['superficie', 'habitaciones'])
df['precio'] = y_array
df.head()

### 1.2 · Exploración

In [ ]:
# Tu código aquí

In [ ]:
df.describe()
df.info()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df['superficie'], df['precio'], alpha=0.7)
axes[0].set_xlabel('superficie')
axes[0].set_ylabel('precio')

axes[1].scatter(df['habitaciones'], df['precio'], alpha=0.7, color='orange')
axes[1].set_xlabel('habitaciones')
axes[1].set_ylabel('precio')
plt.tight_layout()
plt.show()

### 1.3 · Separar X e y

In [ ]:
# Tu código aquí

In [ ]:
X = df.drop(columns='precio')
y = df['precio']

print('X:', X.shape, '· y:', y.shape)

### 1.4 · Train / test split

In [ ]:
# Tu código aquí

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train:', X_train.shape, '· Test:', X_test.shape)

### 1.5 · Entrenar el modelo

In [ ]:
# Tu código aquí

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

### 1.6 · Predecir y evaluar

In [ ]:
# Tu código aquí

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print('MAE :', round(mae, 2))
print('MSE :', round(mse, 2))
print('RMSE:', round(np.sqrt(mse), 2))
print('R²  :', round(r2, 3))

### 1.7 · Gráfico: real vs predicho
Si el modelo es bueno, los puntos se alinean con la diagonal.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, '--', color='red', label='predicción perfecta')
plt.xlabel('Precio real')
plt.ylabel('Precio predicho')
plt.title(f'Real vs predicho · R² = {r2:.3f}')
plt.legend()
plt.show()

### 1.8 · ¿Y si usamos solo una variable?

In [ ]:
for col in ['superficie', 'habitaciones']:
    Xi = df[[col]]
    Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, y, test_size=0.2, random_state=42)
    mi = LinearRegression().fit(Xi_tr, yi_tr)
    print(f'R² con {col:12s}: {r2_score(yi_te, mi.predict(Xi_te)):.3f}')

print(f'R² con las 2       : {r2:.3f}')

### 1.9 · Mini práctica

1. Entrena el modelo con `test_size=0.4`. ¿Bajan las métricas?
2. Cambia `noise=60` en `make_regression` y vuelve a entrenar. ¿Qué le pasa al R² y al gráfico real-vs-predicho?

In [ ]:
# Tu código aquí

---

## Sección 2 · Clasificación · Flores Iris

**Objetivo:** predecir la especie (`setosa`, `versicolor`, `virginica`) a partir de las medidas del sépalo y el pétalo.

Es un problema **multiclase** (3 clases) y los datos están limpios: la regresión logística debería acertar casi todos.

### 2.1 · Imports y datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.head()

### 2.2 · Exploración

In [ ]:
# Tu código aquí

In [ ]:
print('Clases:', iris.target_names)
print('Features:', iris.feature_names)
print('Shape :', df.shape)

print(df['target'].value_counts().sort_index())

# Dispersión 2D con petal length / petal width (las dos más informativas)
fig, ax = plt.subplots(figsize=(6, 5))
for clase, color in zip(iris.target_names, ['#d62728', '#2ca02c', '#1f77b4']):
    sub = df[df['target'] == list(iris.target_names).index(clase)]
    ax.scatter(sub['petal length (cm)'], sub['petal width (cm)'],
               label=clase, alpha=0.8, s=40, color=color, edgecolor='k')
ax.set_xlabel('petal length (cm)')
ax.set_ylabel('petal width (cm)')
ax.set_title('Iris · 2 features más informativas')
ax.legend()
plt.show()

### 2.3 · Separar X e y

In [ ]:
# Tu código aquí

In [ ]:
X = df.drop(columns='target')
y = df['target']

### 2.4 · Train / test split

In [ ]:
# Tu código aquí

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 2.5 · Entrenar el modelo

In [ ]:
# Tu código aquí

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

### 2.6 · Predecir y evaluar

In [ ]:
# Tu código aquí

In [ ]:
y_pred = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print()
print('Matriz de confusión (filas=real, columnas=predicho):')
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### 2.7 · Gráfico: frontera de decisión
Usamos solo 2 features para poder dibujar la frontera. Las clases se separan con rectas.

In [ ]:
X2 = df[['petal length (cm)', 'petal width (cm)']].values
y2 = df['target'].values

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)
m2 = LogisticRegression(max_iter=1000).fit(X2_tr, y2_tr)
acc2 = accuracy_score(y2_te, m2.predict(X2_te))

x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = m2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

cmap_bg   = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_pts  = ListedColormap(['#d62728', '#2ca02c', '#1f77b4'])

plt.figure(figsize=(7, 5))
plt.contourf(xx, yy, Z, alpha=0.35, cmap=cmap_bg)
plt.scatter(X2[:, 0], X2[:, 1], c=y2, cmap=cmap_pts, edgecolor='k', s=40)
plt.xlabel('petal length (cm)')
plt.ylabel('petal width (cm)')
plt.title(f'Frontera de decisión · accuracy (2 features) = {acc2:.3f}')
plt.show()

### 2.8 · ¿Y si usamos solo una variable?

In [ ]:
for col in iris.feature_names:
    Xi = df[[col]]
    Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
        Xi, y, test_size=0.2, random_state=42, stratify=y
    )
    mi = LogisticRegression(max_iter=1000).fit(Xi_tr, yi_tr)
    print(f'Accuracy con {col:20s}: {accuracy_score(yi_te, mi.predict(Xi_te)):.3f}')

print(f'Accuracy con las 4 features  : {accuracy_score(y_test, model.predict(X_test)):.3f}')

### 2.9 · Mini práctica

1. Predice la especie de una flor con `petal length=5.1, petal width=1.8, sepal length=6.5, sepal width=2.8`. ¿Qué especie dice el modelo?
2. Cambia `random_state=0` en el split. ¿Se mantiene el accuracy? ¿Por qué sí o por qué no?

In [ ]:
# Tu código aquí

---

## Sección 3 · Clasificación binaria · Cáncer de mama

**Objetivo:** predecir si un tumor es **maligno** (1) o **benigno** (0) a partir de 30 medidas celulares.

Es un problema **binario** (2 clases). Verás cómo cambia la matriz de confusión y por qué aquí `accuracy` no cuenta toda la historia.

### 3.1 · Imports y datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df['target'] = data.target
df.head()

### 3.2 · Exploración rápida

In [ ]:
print('Shape  :', df.shape)
print('Clases :', data.target_names)
print('Balance:', df['target'].value_counts().to_dict())
print()
print('Con 30 features ya no cabe un scatter de a pares; usaremos PCA para visualizar.')

### 3.3 · Separar X e y, split y escalar
Con 30 features de escalas muy distintas **es obligatorio escalar** antes de LogisticRegression.

In [ ]:
X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

### 3.4 · Entrenar

In [ ]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train_sc, y_train)

### 3.5 · Métricas

In [ ]:
y_pred  = model.predict(X_test_sc)
y_score = model.predict_proba(X_test_sc)[:, 1]

print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('ROC AUC  :', round(roc_auc_score(y_test, y_score), 3))
print()
print('Matriz de confusión (filas=real, columnas=predicho):')
cm = confusion_matrix(y_test, y_pred)
print(cm)
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

### 3.6 · Gráficos: matriz de confusión, curva ROC y proyección PCA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Matriz de confusión
axes[0].imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    axes[0].text(j, i, v, ha='center', va='center', fontsize=14, color='black')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(data.target_names)
axes[0].set_yticklabels(data.target_names)
axes[0].set_xlabel('Predicho'); axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de confusión')

# Curva ROC
RocCurveDisplay.from_estimator(model, X_test_sc, y_test, ax=axes[1])
axes[1].set_title('Curva ROC')
axes[1].plot([0, 1], [0, 1], '--', color='gray')

# Proyección PCA 2D coloreada por clase
pca = PCA(n_components=2, random_state=42)
X_all_sc = scaler.transform(X)
Xp = pca.fit_transform(X_all_sc)
for clase, color in zip(data.target_names, ['#d62728', '#1f77b4']):
    mask = (y == list(data.target_names).index(clase))
    axes[2].scatter(Xp[mask, 0], Xp[mask, 1], label=clase, alpha=0.6, s=20, color=color)
axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')
axes[2].set_title(f'PCA 2D · varianza explicada = {pca.explained_variance_ratio_.sum():.2f}')
axes[2].legend()

plt.tight_layout()
plt.show()

### 3.7 · Cierre

| | Iris (sección 2) | Cáncer de mama (sección 3) |
|---|---|---|
| Tipo | Multiclase (3) | Binaria (2) |
| Features | 4 (mismas escalas) | 30 (escalas muy distintas) |
| Preproceso | Ninguno | `StandardScaler` obligatorio |
| Métrica principal | Accuracy | Accuracy + ROC AUC |
| Matriz de confusión | 3×3 | 2×2 (TN, FP, FN, TP) |

**Idea clave:** en clasificación binaria, predecir siempre la clase mayoritaria puede dar un accuracy alto sin servir de nada. Por eso miramos también la matriz de confusión, precisión/recall por clase y el área bajo la curva ROC.